# Baltic Sea ice animation

This notebook creates a looping GIF from the 2020/21?2024/25 chart record. Map colors show sea-ice concentration; the moving lower-panel marker follows the supplied basin-wide extent series.


In [ ]:
from pathlib import Path
import sys
import os

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Start Jupyter from this repository or one of its subfolders.")

PROJECT_ROOT = find_project_root()
(PROJECT_ROOT / ".cache" / "matplotlib").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".cache" / "matplotlib"))
(PROJECT_ROOT / ".cache" / "cartopy").mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.animation import PillowWriter, FuncAnimation
import numpy as np
import pandas as pd
import xarray as xr
import cartopy
cartopy.config["data_dir"] = str(PROJECT_ROOT / ".cache" / "cartopy" )
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from IPython.display import Image, display

In [ ]:
WINTER_YEARS = range(2020, 2025)
EXTENT_PATH = PROJECT_ROOT / "data" / "raw" / "baltic_ice_extent_full_record.nc"
SIC_DIRECTORY = PROJECT_ROOT / "data" / "raw" / "recent_baltic_winter_subsets"
SIC_FILES = {
    year: SIC_DIRECTORY / f"baltic_ice_concentration_winter_{year}_{year + 1}.nc"
    for year in WINTER_YEARS
}

FRAME_EVERY_N_VALID_CHARTS = 7
DISPLAY_STRIDE = 2
ICE_EXTENT_THRESHOLD_PCT = 15.0
FRAMES_PER_SECOND = 6
OUTPUT_DIR = PROJECT_ROOT / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GIF_PATH = OUTPUT_DIR / "baltic_ice_atlas_recent_winters.gif"
COVER_PATH = OUTPUT_DIR / "baltic_ice_atlas_cover.png"
MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "animation_frame_manifest.csv"
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

missing = [str(p) for p in [EXTENT_PATH, *SIC_FILES.values()] if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Required local files are missing. This notebook never downloads them:\n" + "\n".join(missing)
    )
print(f"Using {len(SIC_FILES)} existing winter subsets; no data retrieval is enabled.")

## Load the historical extent series

The supplied basin series is in millions of square kilometres. The figure converts it to thousand km^2 and retains the archive's weekly-to-daily sampling change. It is a chart-derived product extent series, not a value recomputed from the SIC grid.

In [ ]:
with xr.open_dataset(EXTENT_PATH, decode_cf=True, mask_and_scale=True) as extent_ds:
    extent_units = extent_ds["sea_ice_extent"].attrs.get("units")
    if extent_units != "1e6km2":
        raise ValueError(f"Expected sea_ice_extent units '1e6km2', got {extent_units!r}")
    extent_dates_all = pd.DatetimeIndex(pd.to_datetime(extent_ds.time.values))
    extent_values_all = np.asarray(extent_ds["sea_ice_extent"].values, dtype=float) * 1000.0

extent_valid = np.isfinite(extent_values_all)
extent_dates = extent_dates_all[extent_valid]
extent_values_1000km2 = extent_values_all[extent_valid]
if len(extent_dates) == 0:
    raise ValueError("No valid sea_ice_extent observations were found.")
print(f"Extent observations: {len(extent_dates):,}; {extent_dates.min():%Y-%m-%d} to {extent_dates.max():%Y-%m-%d}")

## Prepare approximately weekly map frames

To keep the video compact, use every seventh valid chart in each winter and display every second grid cell. The source data remain at their native grid and date resolution. The animation caption shows each actual chart date; it does not interpolate between maps.

In [ ]:
frame_dates = []
frame_fields = []
frame_winter_years = []
longitude = latitude = None

for year, path in SIC_FILES.items():
    with xr.open_dataset(
        path,
        chunks={"time": 30, "latitude": 256, "longitude": 256},
        decode_cf=True,
        mask_and_scale=True,
    ) as ds:
        if "ice_concentration" not in ds:
            raise KeyError(f"{path.name} does not contain ice_concentration")
        sic = ds["ice_concentration"]
        dates = pd.DatetimeIndex(pd.to_datetime(ds.time.values))
        labels = np.where(dates.month <= 5, dates.year - 1, dates.year)
        winter_indexes = np.flatnonzero(labels == year)
        if not winter_indexes.size:
            raise ValueError(f"No dates for winter {year}/{year + 1} in {path.name}")

        sampled = sic.isel(
            time=winter_indexes,
            latitude=slice(None, None, DISPLAY_STRIDE),
            longitude=slice(None, None, DISPLAY_STRIDE),
        )
        has_observation = sampled.notnull().any(("latitude", "longitude")).compute().values
        observed_local_indexes = np.flatnonzero(has_observation)
        chosen_local_indexes = observed_local_indexes[::FRAME_EVERY_N_VALID_CHARTS]
        if not chosen_local_indexes.size:
            raise ValueError(f"No valid SIC maps for winter {year}/{year + 1}")

        chosen = sampled.isel(time=chosen_local_indexes).load()
        current_latitude = np.asarray(chosen.latitude.values, dtype=float)
        current_longitude = np.asarray(chosen.longitude.values, dtype=float)
        if latitude is None:
            latitude, longitude = current_latitude, current_longitude
        elif not (
            np.array_equal(latitude, current_latitude)
            and np.array_equal(longitude, current_longitude)
        ):
            raise ValueError("Winter subsets have different spatial coordinates.")

        frame_dates.extend(pd.DatetimeIndex(pd.to_datetime(chosen.time.values)))
        frame_fields.extend(np.asarray(chosen.values, dtype=np.float32))
        frame_winter_years.extend([year] * chosen.sizes["time"])
        print(f"Prepared winter {year}/{year + 1}: {chosen.sizes['time']} frames")

frame_dates = pd.DatetimeIndex(frame_dates)
frame_fields = np.stack(frame_fields)
frame_winter_years = np.asarray(frame_winter_years, dtype=int)

order = np.argsort(frame_dates.values)
frame_dates = frame_dates[order]
frame_fields = frame_fields[order]
frame_winter_years = frame_winter_years[order]

frame_manifest = pd.DataFrame({
    "date": frame_dates,
    "winter_start_year": frame_winter_years,
    "sic_threshold_for_extent_pct": ICE_EXTENT_THRESHOLD_PCT,
})
frame_manifest.to_csv(MANIFEST_PATH, index=False)
print(f"Animation frames: {len(frame_dates)}; downsampled field shape: {frame_fields.shape[1:]}")

## Build the map and synchronized time series

Country labels are placed manually for legibility. Cartopy supplies coastlines and national boundaries. On first use, Cartopy may fetch its small Natural Earth boundary files into its local cache; this is map context only, not sea-ice data.

In [ ]:
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titleweight": "bold",
})
map_crs = ccrs.LambertConformal(
    central_longitude=20,
    central_latitude=59,
    standard_parallels=(55, 65),
)
data_crs = ccrs.PlateCarree()

fig = plt.figure(figsize=(8, 10), facecolor="#f7f8fa")
grid = fig.add_gridspec(2, 1, height_ratios=[3.2, 1.15], hspace=0.17)
map_ax = fig.add_subplot(grid[0], projection=map_crs)
series_ax = fig.add_subplot(grid[1])

map_ax.set_extent([9, 31, 53.2, 66.2], crs=data_crs)
map_ax.set_facecolor("#dceaf0")
map_ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="#eeeae2", edgecolor="none", zorder=3)
map_ax.add_feature(cfeature.COASTLINE.with_scale("50m"), edgecolor="#52616b", linewidth=0.55, zorder=4)
map_ax.add_feature(cfeature.BORDERS.with_scale("50m"), edgecolor="#8a9298", linewidth=0.45, zorder=4)

label_style = dict(
    transform=data_crs, ha="center", va="center", fontsize=8,
    color="#4b545b", weight="semibold", zorder=5,
)
country_labels = {
    "SWEDEN": (17.0, 61.7),
    "FINLAND": (26.0, 63.4),
    "RUSSIA": (29.6, 60.2),
    "ESTONIA": (25.7, 58.8),
    "LATVIA": (24.8, 56.9),
    "LITHUANIA": (23.5, 55.3),
    "POLAND": (18.7, 54.5),
    "GERMANY": (13.0, 54.4),
    "DENMARK": (10.8, 56.0),
}
for label, (lon, lat) in country_labels.items():
    map_ax.text(lon, lat, label, **label_style)

gridlines = map_ax.gridlines(
    crs=data_crs, draw_labels=False,
    linewidth=0.35, color="#8ba0ab", alpha=0.45, linestyle=":",
)
map_ax.set_title("Baltic Sea ice concentration", loc="left", fontsize=15, pad=8)

first_field = np.ma.masked_invalid(frame_fields[0])
mesh = map_ax.pcolormesh(
    longitude, latitude, first_field,
    cmap="Blues", vmin=0, vmax=100, shading="auto",
    transform=data_crs, zorder=2,
)
colorbar = fig.colorbar(mesh, ax=map_ax, orientation="vertical", pad=0.025, shrink=0.78)
colorbar.set_label("Sea-ice concentration (%)")
colorbar.set_ticks([0, 15, 30, 50, 75, 100])

threshold_contour = None
if np.nanmin(frame_fields[0]) < ICE_EXTENT_THRESHOLD_PCT < np.nanmax(frame_fields[0]):
    threshold_contour = map_ax.contour(
        longitude, latitude, frame_fields[0],
        levels=[ICE_EXTENT_THRESHOLD_PCT],
        colors=["#e4572e"], linewidths=1.0,
        transform=data_crs, zorder=5,
    )
map_ax.plot([], [], color="#e4572e", linewidth=1.4, label="15% SIC ice-extent edge")
map_ax.legend(loc="lower left", frameon=True, framealpha=0.9, fontsize=8)

weekly_mask = extent_dates < pd.Timestamp("2006-12-01")
daily_mask = ~weekly_mask
series_ax.scatter(
    extent_dates[weekly_mask], extent_values_1000km2[weekly_mask],
    s=7, color="#276A8A", label="Weekly charts (through 2005/06)", zorder=2,
)
series_ax.scatter(
    extent_dates[daily_mask], extent_values_1000km2[daily_mask],
    s=5, color="#C85C3D", label="Daily charts (from 2006/07)", zorder=2,
)
series_ax.axvline(
    pd.Timestamp("2006-12-01"), color="#555555", linestyle="--", linewidth=0.8, zorder=1,
)
series_ax.set_xlim(pd.Timestamp("1980-01-01"), pd.Timestamp("2026-12-31"))
series_ax.set_ylim(bottom=0)
series_ax.set_ylabel("Extent (thousand km^2)")
series_ax.set_xlabel("Date")
series_ax.grid(axis="y", color="#d7dee2", linewidth=0.7)
series_ax.spines[["top", "right"]].set_visible(False)
series_ax.xaxis.set_major_locator(mdates.YearLocator(5))
series_ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
series_ax.set_title("Historical Baltic basin sea-ice extent (observed charts)", loc="left", fontsize=11, pad=5)
series_ax.legend(loc="upper right", frameon=False, fontsize=8, ncol=2)

first_nearest = int(np.argmin(np.abs((extent_dates - frame_dates[0]).days)))
time_cursor = series_ax.axvline(frame_dates[0], color="#e4572e", linewidth=1.3, alpha=0.9)
extent_cursor, = series_ax.plot(
    [extent_dates[first_nearest]], [extent_values_1000km2[first_nearest]],
    marker="o", markersize=7, color="#e4572e", markeredgecolor="white",
    markeredgewidth=0.8, zorder=5,
)
date_label = map_ax.text(
    0.99, 0.98, "", transform=map_ax.transAxes, ha="right", va="top",
    color="#1b252b", fontsize=11, weight="bold",
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="none", alpha=0.9),
    zorder=6,
)
winter_label = fig.text(0.01, 0.01, "Copernicus Marine Baltic Sea Ice MY (DOI: 10.48670/moi-00131)", fontsize=8, color="#59636a")

def update_frame(index):
    global threshold_contour
    field = np.ma.masked_invalid(frame_fields[index])
    mesh.set_array(field.ravel())
    if threshold_contour is not None:
        threshold_contour.remove()
        threshold_contour = None
    finite = frame_fields[index][np.isfinite(frame_fields[index])]
    if finite.size and finite.min() < ICE_EXTENT_THRESHOLD_PCT < finite.max():
        threshold_contour = map_ax.contour(
            longitude, latitude, frame_fields[index],
            levels=[ICE_EXTENT_THRESHOLD_PCT],
            colors=["#e4572e"], linewidths=1.0,
            transform=data_crs, zorder=5,
        )

    current_date = frame_dates[index]
    current_winter = frame_winter_years[index]
    date_label.set_text(
        f"Winter {current_winter}/{current_winter + 1} | {current_date:%d %b %Y}"
    )
    nearest = int(np.argmin(np.abs((extent_dates - current_date).days)))
    time_cursor.set_xdata([current_date, current_date])
    extent_cursor.set_data([extent_dates[nearest]], [extent_values_1000km2[nearest]])
    return mesh, time_cursor, extent_cursor, date_label

animation = FuncAnimation(
    fig, update_frame, frames=len(frame_dates),
    interval=1000 / FRAMES_PER_SECOND, blit=False, repeat=True,
)
ice_pixels_per_frame = np.sum(frame_fields >= ICE_EXTENT_THRESHOLD_PCT, axis=(1, 2))
cover_index = int(np.argmax(ice_pixels_per_frame))
update_frame(cover_index)
fig.savefig(COVER_PATH, dpi=144, facecolor=fig.get_facecolor())
print(f"Poster frame saved: {COVER_PATH} ({frame_dates[cover_index]:%Y-%m-%d})")
print(f"Timeline cursor is matched to the nearest available basin extent observation.")
display(Image(filename=str(COVER_PATH)))

## Export the GIF

The output is a looping 4:5 GIF at 576 by 720 pixels and 6 frames per second. Pillow is included with Matplotlib. Rendering takes a few minutes because each frame includes a gridded map and country boundaries.


In [ ]:
animation.save(GIF_PATH, writer=PillowWriter(fps=FRAMES_PER_SECOND), dpi=72)
print(f"GIF saved: {GIF_PATH}")
print(f"Frames: {len(frame_dates)} | fps: {FRAMES_PER_SECOND} | duration: {len(frame_dates) / FRAMES_PER_SECOND:.1f} s")
display(Image(filename=str(GIF_PATH)))


## Reading the animation

- Color shows SIC, not ice thickness.
- The orange line is the SIC >= 15% edge used to define mapped sea-ice extent.
- The lower-panel line is the supplied basin-level extent series in thousand km^2; the moving marker uses its nearest valid observation for each SIC chart date.
- Maps cover only the five recent complete winters available locally. The historical lower panel provides longer context, but the animation does not imply that full-period basin-wide maps are available.
- Approximately weekly frames show actual chart observations; no values are interpolated across dates.